### LangChain

In [1]:
# Run this once in the notebook to install the required packages.
# %pip install -U langchain langchain-openai langchain-text-splitters

from pathlib import Path
import httpx2
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from rich import print
from pydantic import SecretStr
from langchain.agents import create_agent
from typing import List
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownTextSplitter, MarkdownHeaderTextSplitter

C:\Users\marij\AppData\Local\Temp\ipykernel_28528\2718659266.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
def print_conversation(messages:List[BaseMessage]) -> None:
    for message in messages:
       message.pretty_print() 


In [3]:
openai_api_key = SecretStr(Path('openai-secret-key-ai-integrations-developers.txt').read_text(encoding='utf-8').strip())
openai_model = ChatOpenAI(
    model_name='gpt-5-nano',
    openai_api_key=openai_api_key,
    reasoning_effort='low',
    http_client=httpx2.Client(trust_env=False),
)

### Short-term Memory

In [4]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver keeps each thread's messages only while this notebook kernel runs.
memory = InMemorySaver()
agent = create_agent(
    model=openai_model,
    tools=[],
    system_prompt="You are a helpful assistant.",
    checkpointer=memory,
)

# This ID identifies one separate conversation.
thread_1 = {"configurable": {"thread_id": "thread_1"}}


In [5]:
# First message in thread_1: the checkpointer saves it.
result_1 = agent.invoke(
    {"messages": [HumanMessage(content="Hello, AI! My name is Maria.")]},
    config=thread_1,
)
print(result_1["messages"][-1].content)


Hi Maria! Nice to meet you. How can I help you today?

In [6]:
# Same thread ID: the agent receives the earlier message automatically.
result_2 = agent.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=thread_1,
)
print(result_2["messages"][-1].content)  # The answer should be Tony.


Your name is Maria. Nice to meet you again—how can I help you today?

### Long-term Memory

In [7]:
from langgraph.store.memory import InMemoryStore

In [8]:
# Create these once per kernel session. Do not rerun this cell between saving and recalling facts.
store = InMemoryStore()
checkpointer = InMemorySaver()

### Long-term memory tools

In [9]:
# Pydantic on Python 3.11 requires TypedDict from typing_extensions.
from typing_extensions import TypedDict
from langchain.tools import ToolRuntime, tool


# The agent receives this value at invocation time. It keeps each user's facts separate.
class CustomAgentContext(TypedDict):
    user_id: str


@tool
def remember_user_facts(
    key: str,
    value: str,
    runtime: ToolRuntime[CustomAgentContext],
) -> str:
    """Extract durable user facts from a user message and store them in long-term memory.

    Example: key='allergy', value='The user is allergic to nuts.'

    Args:
        key: A unique identifier for the fact.
        value: The fact itself.
    """
    namespace = ("users", runtime.context["user_id"], "general_knowledge")
    previous_item = runtime.store.get(namespace, "auto_extracted_facts")

    facts_dict = previous_item.value if previous_item is not None else {}
    facts_dict[key] = value

    runtime.store.put(namespace, "auto_extracted_facts", facts_dict)
    return "OK"


@tool
def recall_user_facts(runtime: ToolRuntime[CustomAgentContext]) -> str:
    """Recall previously stored long-term facts about the user."""
    namespace = ("users", runtime.context["user_id"], "general_knowledge")
    results = runtime.store.search(namespace, limit=20)

    if not results:
        return "No facts stored."

    return "\n---\n".join(
        f"{facts_group.key}:\n"
        + "\n".join(
            f"- {key}: {value}" for key, value in facts_group.value.items()
        )
        for facts_group in results
    )


### Agent with long-term user facts

In [10]:
from langchain_core.runnables import RunnableLambda


# Reuse the one store and checkpointer created above.

long_term_memory_agent = create_agent(
    model=openai_model,
    tools=[remember_user_facts, recall_user_facts],
    system_prompt=f"""
You are a polite and helpful personal assistant.
At the start of every interaction, call `{recall_user_facts.name}` to check
whether facts about this user have already been stored.
When the user provides a durable fact (for example a name, hobby, plan,
need, preference, or location), call `{remember_user_facts.name}` after
recalling and before your final answer. Do this even when the user does not
explicitly say the word 'remember'.
Be friendly and use known facts when they are relevant.
""",
    checkpointer=checkpointer,
    store=store,
    context_schema=CustomAgentContext,
)

# This runnable prints the complete message history after each agent invocation.
interact = long_term_memory_agent | RunnableLambda(
    lambda result: print_conversation(result["messages"])
)


In [11]:
# This call recalls first, then saves the name, city, and hobby in `store`.
save_result = long_term_memory_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="My name is Maria, I live in Sofia, and I enjoy hiking."
            )
        ]
    },
    config={"configurable": {"thread_id": "maria-thread-1"}},
    context={"user_id": "maria"},
)
print_conversation(save_result["messages"])


================================ Human Message =================================

My name is Maria, I live in Sofia, and I enjoy hiking.
================================== Ai Message ==================================
Tool Calls:
  recall_user_facts (call_Y4QwuHhqNjZWBNGbOeddGjcX)
 Call ID: call_Y4QwuHhqNjZWBNGbOeddGjcX
  Args:
================================= Tool Message =================================
Name: recall_user_facts

No facts stored.
================================== Ai Message ==================================
Tool Calls:
  remember_user_facts (call_PUMuPvnpyGCncSsw326xEJ4k)
 Call ID: call_PUMuPvnpyGCncSsw326xEJ4k
  Args:
    key: name
    value: Maria
  remember_user_facts (call_yLePixSOUfGvUo8GYcG1QR9e)
 Call ID: call_yLePixSOUfGvUo8GYcG1QR9e
  Args:
    key: location
    value: Sofia
  remember_user_facts (call_U6rUT3G3L2NcogZe92vxpyj2)
 Call ID: call_U6rUT3G3L2NcogZe92vxpyj2
  Args:
    key: hobby
    value: hiking
================================= Tool Message ==

In [12]:
# A new thread has no chat history, but the same user_id reads facts from the same store.
recall_result = long_term_memory_agent.invoke(
    {"messages": [HumanMessage(content="What do you remember about me?")]},
    config={"configurable": {"thread_id": "maria-thread-2"}},
    context={"user_id": "maria"},
)
print_conversation(recall_result["messages"])


================================ Human Message =================================

What do you remember about me?
================================== Ai Message ==================================
Tool Calls:
  recall_user_facts (call_K2SpNvAlNZFFa8UJNDRr73gz)
 Call ID: call_K2SpNvAlNZFFa8UJNDRr73gz
  Args:
================================= Tool Message =================================
Name: recall_user_facts

auto_extracted_facts:
- name: Maria
- location: Sofia
- hobby: hiking
================================== Ai Message ==================================

Here’s what I remember about you:
- Name: Maria
- Location: Sofia
- Hobby: hiking

If any of that isn’t accurate, or you want to add more details (e.g., profession, favorite foods, goals), tell me and I can store it as a durable fact.


### Custom Agent State

In [13]:
from langchain.agents.middleware import AgentState, after_model
from langgraph.runtime import Runtime


@after_model
def print_state_after_model(state: AgentState, runtime: Runtime) -> None:
    """Show the complete agent state after each model response."""
    print("\nAfter model:")
    print(state)


class TrackUsageAgentState(AgentState):
    input_tokens: int
    cached_tokens: int
    output_tokens: int
    reasoning_tokens: int


@after_model(state_schema=TrackUsageAgentState)
def track_usage(
    state: TrackUsageAgentState,
    runtime: Runtime,
) -> dict:
    """Read token usage from the latest AI message and store totals in state."""
    last_message = state["messages"][-1]
    usage = last_message.usage_metadata or {}

    input_tokens = usage.get("input_tokens", 0)
    cached_tokens = usage.get("input_token_details", {}).get("cache_read", 0)
    output_tokens = usage.get("output_tokens", 0)
    reasoning_tokens = usage.get("output_token_details", {}).get("reasoning", 0)

    print(
        f"Input tokens: {input_tokens} ({cached_tokens} cached); "
        f"Output tokens: {output_tokens} ({reasoning_tokens} reasoning)"
    )

    # Returning these values makes them part of the agent state.
    return {
        "input_tokens": state.get("input_tokens", 0) + input_tokens,
        "cached_tokens": state.get("cached_tokens", 0) + cached_tokens,
        "output_tokens": state.get("output_tokens", 0) + output_tokens,
        "reasoning_tokens": state.get("reasoning_tokens", 0) + reasoning_tokens,
    }

In [14]:
usage_agent = create_agent(
    model=openai_model,
    tools=[],
    middleware=[
        print_state_after_model,
        track_usage,
    ],
    state_schema=TrackUsageAgentState,
)

In [15]:
usage_result = usage_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Explain why the square root of 2 is irrational."
            )
        ]
    },
    config={
        "configurable": {
            "thread_id": "usage-demo-1"
        }
    },
)

print("\nFinal answer:")
print(usage_result["messages"][-1].content)

print("\nUsage tracked in agent state:")
print(f"Input tokens: {usage_result['input_tokens']}")
print(f"Cached tokens: {usage_result['cached_tokens']}")
print(f"Output tokens: {usage_result['output_tokens']}")
print(f"Reasoning tokens: {usage_result['reasoning_tokens']}")

Input tokens: 17 (0 cached); Output tokens: 251 (64 reasoning)

After model:

{
    'messages': [
        HumanMessage(
            content='Explain why the square root of 2 is irrational.',
            additional_kwargs={},
            response_metadata={},
            id='f3908204-2f53-4751-8c9e-1c167ab5124d'
        ),
        AIMessage(
            content='A classic proof by contradiction.\n\n- Suppose √2 is rational. Then √2 = a/b for integers a, b
with gcd(a, b) = 1 (the fraction is in lowest terms).\n- Squaring both sides gives 2 = a^2 / b^2, so a^2 = 2b^2.\n-
Therefore a^2 is even, which implies a is even. Write a = 2k for some integer k.\n- Substitute back: (2k)^2 = 2b^2 
→ 4k^2 = 2b^2 → b^2 = 2k^2, so b^2 is even, hence b is even.\n- Thus both a and b are even, contradicting the 
assumption that a/b was in lowest terms.\n\nTherefore our initial assumption is false, and √2 is irrational.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 251,
                    'prompt_tokens': 17,
                    'total_tokens': 268,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-nano-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ELYWWlMkwJ1mKZiMgyS2uJVCjL3he',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a07d2a-157d-7f60-a72b-3c9d6db56075-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 17,
                'output_tokens': 251,
                'total_tokens': 268,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        )
    ],
    'input_tokens': 17,
    'cached_tokens': 0,
    'output_tokens': 251,
    'reasoning_tokens': 64
}

Final answer:

A classic proof by contradiction.

- Suppose √2 is rational. Then √2 = a/b for integers a, b with gcd(a, b) = 1 (the fraction is in lowest terms).
- Squaring both sides gives 2 = a^2 / b^2, so a^2 = 2b^2.
- Therefore a^2 is even, which implies a is even. Write a = 2k for some integer k.
- Substitute back: (2k)^2 = 2b^2 → 4k^2 = 2b^2 → b^2 = 2k^2, so b^2 is even, hence b is even.
- Thus both a and b are even, contradicting the assumption that a/b was in lowest terms.

Therefore our initial assumption is false, and √2 is irrational.

Usage tracked in agent state:

Input tokens: 17

Cached tokens: 0

Output tokens: 251

Reasoning tokens: 64

### Context Management

In [16]:
from langchain.agents.middleware import SummarizationMiddleware, after_model

In [17]:
summary_agent = create_agent(
    model=openai_model,
    system_prompt="You are a helpful assistant that summarizes conversations.",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=openai_model,
            trigger=("tokens", 1000),
            keep=("messages", 10),
        )
    ],
)

In [18]:
summary_config = {
    "configurable": {
        "thread_id": "summary-test"
    }
}

In [19]:
messages_to_send = [
    "My name is Maria and I live in Sofia.",
    "I am learning LangChain agents.",
    "My favorite hobby is hiking.",
    "I want to build a helpful FAQ assistant.",
]

for text in messages_to_send:
    result = summary_agent.invoke(
        {"messages": [HumanMessage(content=text)]},
        config=summary_config,
    )
    
print_conversation(result["messages"])

================================ Human Message =================================

My name is Maria and I live in Sofia.
================================== Ai Message ==================================

Nice to meet you, Maria. You’ve told me that you live in Sofia. If you’d like, I can help summarize any details you share or assist with tasks related to Sofia or anything else. How can I help today?
================================ Human Message =================================

I am learning LangChain agents.
================================== Ai Message ==================================

Nice to meet you, Maria. Since you’re learning LangChain agents, here’s a quick roadmap and some starter pointers to help you get going.

What LangChain agents are (in brief)
- Agents: autonomous modules that can decide what actions to take (e.g., call a tool, search the web, fetch data) based on a goal and a plan.
- Tools/Tools integration: external capabilities the agent can use (calculator, searc

### Context editing for long tool-using conversations

Large tool results, such as logs, can quickly fill a model's context window. `ContextEditingMiddleware` clears older tool outputs once the conversation exceeds a token threshold, while keeping recent tool results available.

In [20]:
from collections.abc import Callable
from langchain.agents.middleware import (
    ClearToolUsesEdit,
    ContextEditingMiddleware,
    ModelRequest,
    ModelResponse,
    wrap_model_call,
)


@tool
def fetch_logs(service: str) -> str:
    """Fetch raw application logs for the specified service."""
    # Deliberately return many lines: this simulates a large, expensive tool result.
    lines = [
        (
            f"2026-04-20T10:15:{second:02d}Z service={service} "
            f"level=ERROR request_id=abc123 user_id=user-{second:04d} "
            "message=Database connection timed out"
        )
        for second in range(60)
    ]
    return "\n".join(lines)


@wrap_model_call
def log_model_input(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """Print a compact preview of the messages that will reach the model."""
    print(f"\n--- Model sees {len(request.messages)} message(s) ---")
    previews = []
    for message in request.messages:
        preview = str(getattr(message, "content", ""))[:60].replace("\n", " ")
        tool_calls = getattr(message, "tool_calls", [])
        extra = f" tool_calls={len(tool_calls)}" if tool_calls else ""
        previews.append(f"- {message.type}: {preview}{extra}")
    print(" | ".join(previews))
    return handler(request)


context_editing_agent = create_agent(
    model=openai_model,
    tools=[fetch_logs],
    system_prompt=(
        f"You are an on-call SRE assistant. For every incident question, "
        f"first call `{fetch_logs.name}` for the relevant service, then summarize "
        "the likely root cause."
    ),
    checkpointer=InMemorySaver(),
    middleware=[
        log_model_input,
        ContextEditingMiddleware(
            edits=[
                # When the context exceeds 500 tokens, keep the latest tool result
                # and replace older log outputs with a small placeholder.
                ClearToolUsesEdit(trigger=500, keep=1, clear_tool_inputs=True)
            ]
        ),
    ],
)


In [21]:
# The second incident adds another large log result. Watch the model-input output:
# older logs are cleared, while the latest tool result remains available.
sre_config = {"configurable": {"thread_id": "sre-demo"}}

first_incident = context_editing_agent.invoke(
    {"messages": [HumanMessage(content="Investigate an incident in the payments service.")]},
    config=sre_config,
)

second_incident = context_editing_agent.invoke(
    {"messages": [HumanMessage(content="Now investigate an incident in the catalog service.")]},
    config=sre_config,
)

print_conversation(second_incident["messages"])


--- Model sees 1 message(s) ---

- human: Investigate an incident in the payments service.

--- Model sees 3 message(s) ---

- human: Investigate an incident in the payments service. | - ai:  tool_calls=1 | - tool: 2026-04-20T10:15:00Z 
service=payments level=ERROR request_id

--- Model sees 5 message(s) ---

- human: Investigate an incident in the payments service. | - ai:  tool_calls=1 | - tool: 2026-04-20T10:15:00Z 
service=payments level=ERROR request_id | - ai: Summary of logs collected for payments service (timestamp ar | - 
human: Now investigate an incident in the catalog service.

--- Model sees 7 message(s) ---

- human: Investigate an incident in the payments service. | - ai:  tool_calls=1 | - tool: 2026-04-20T10:15:00Z 
service=payments level=ERROR request_id | - ai: Summary of logs collected for payments service (timestamp ar | - 
human: Now investigate an incident in the catalog service. | - ai:  tool_calls=1 | - tool: 2026-04-20T10:15:00Z 
service=catalog level=ERROR request_id=

================================ Human Message =================================

Investigate an incident in the payments service.
================================== Ai Message ==================================
Tool Calls:
  fetch_logs (call_Pj3iW4UB27ECILKwVIsp3hpZ)
 Call ID: call_Pj3iW4UB27ECILKwVIsp3hpZ
  Args:
    service: payments
================================= Tool Message =================================
Name: fetch_logs

2026-04-20T10:15:00Z service=payments level=ERROR request_id=abc123 user_id=user-0000 message=Database connection timed out
2026-04-20T10:15:01Z service=payments level=ERROR request_id=abc123 user_id=user-0001 message=Database connection timed out
2026-04-20T10:15:02Z service=payments level=ERROR request_id=abc123 user_id=user-0002 message=Database connection timed out
2026-04-20T10:15:03Z service=payments level=ERROR request_id=abc123 user_id=user-0003 message=Database connection timed out
2026-04-20T10:15:04Z service=payments level=ERROR request_id=abc12

### Human-in-the-loop

### Travel operations agent

In [ ]:
import json


@tool
def search_travel_options(destination: str) -> str:
    """Search travel options for a destination, including flights and hotels."""
    options = {
        "destination": destination,
        "flights": [
            {"flight_code": "LH170", "departure": "08:10", "price_eur": 189},
            {"flight_code": "FR402", "departure": "09:05", "price_eur": 129},
        ],
        "hotels": [
            {"hotel_name": "Alexander Hub Hotel", "price_eur": 176},
            {"hotel_name": "Spinnel Palace Inn", "price_eur": 144},
        ],
    }
    return json.dumps(options, indent=2)


@tool
def book_flight(traveler_name: str, flight_code: str) -> str:
    """Book a flight for a traveler."""
    return f"Booked flight {flight_code} for {traveler_name}."


@tool
def book_hotel(traveler_name: str, hotel_name: str, nights: int) -> str:
    """Book a hotel for a traveler for a specified number of nights."""
    return f"Booked {nights} night(s) at {hotel_name} for {traveler_name}."


travel_agent = create_agent(
    model=openai_model,
    tools=[search_travel_options, book_flight, book_hotel],
    system_prompt="You are a travel operations assistant. Search before booking.",
    checkpointer=InMemorySaver(),
)
